[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/05.%20The%20Single%20Truck%20Gate%20Entry%20Analysis/P5-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 5. The Single Truck Gate Entry Analysis

## Tier 2 — The Classic Heuristic (Python Implementation)

### Goal
Build and examine a priority-based list scheduling heuristic for assigning arriving trucks to gate booths. The algorithm uses known truck arrivals and service requirements, selects the most urgent truck that has arrived, and assigns it to a gate using a score that accounts for gate availability and service rate.

This is a heuristic: it constructs a feasible schedule using local decisions, but it does not prove that the schedule is optimal. Its purpose is to make the decision process explicit and computationally practical for the example.

### Inputs, outputs, and scope
- **Inputs:** Truck IDs, arrival times, base service times, priorities, and gate service-rate multipliers.
- **Outputs:** Each truck's assigned gate, start time, completion time, and waiting time, together with schedule-level and gate-level metrics.
- **Time unit:** Arrival times, service times, waiting times, and completion times are measured in minutes. Priorities are unitless; a larger value means greater urgency.
- **Scope:** Arrivals and service requirements are known, service is deterministic and non-preemptive, and each truck uses one gate at a time. The model does not include setup times, gate-truck compatibility rules, breakdowns, or uncertainty.

### Learning sequence
1. Represent trucks and gates and define how service rates affect processing duration.
2. Use a priority queue to select the next arrived truck.
3. Score gates and assign the truck to the gate with the lowest score.
4. Calculate schedule metrics, visualize the schedule, and compare the priority rule with an FCFS baseline.

The example contains eight trucks and three gates. The data below are the inputs to the demonstration; the results are produced by running the notebook.

## Model and scheduling logic

### Service duration and gate assignment
A truck's `service_time` is its base processing requirement in minutes. A gate's `service_rate` is a positive multiplier relative to that base: a multiplier above 1 processes the truck faster, and a multiplier below 1 processes it more slowly. For truck $i$ and gate $j$, the effective processing duration is

$$p_{ij} = \frac{s_i}{r_j},$$

where $s_i$ is the truck's base service time and $r_j$ is the gate's service-rate multiplier. Thus, the scheduled duration remains in minutes. The code calculates this duration in `Gate.assign_truck`.

### Priority queue and event times
At time $t$, the active priority queue contains trucks that have arrived by $t$ but have not yet been assigned. The algorithm selects the highest-priority truck from this queue. If no truck is waiting, time advances to the next arrival. Assigning a truck records its future start and completion times; it does not prevent the algorithm from considering another arrival before that completion time.

For truck $i$, the start time on gate $j$ is

$$b_i = \max\{a_i,\ t,\ A_j\},$$

where $a_i$ is its arrival time, $t$ is the current decision time, and $A_j$ is the gate's next available time. Its completion time and waiting time are

$$c_i = b_i + p_{ij}, \qquad w_i = b_i - a_i.$$

The gate's next available time is then updated to $c_i$. These equations are implemented by `Gate.assign_truck`; they ensure that a truck never starts before it arrives or before its assigned gate is available.

### Gate score
For an arrived truck $i$ and gate $j$, the heuristic score is

$$S_{ij}(t) = (11-q_i)\alpha + \max(0, A_j-t)\beta + \frac{0.1}{r_j},$$

where $q_i$ is the truck's priority, $\alpha$ is `priority_weight`, and $\beta$ is `wait_weight`. The algorithm chooses the gate with the lowest score. The priority term is the same for every gate considered for a particular truck, so it does not change that truck's gate ranking; priority determines the order in which trucks are selected from the queue. The waiting and efficiency terms distinguish gates. The score is a heuristic ranking, not a physical cost measured in minutes.

### Schedule metrics
The reported makespan is the last completion time, measured from time zero. For $n$ processed trucks and $m$ gates, average waiting time and average gate utilization are

$$\overline{w} = \frac{1}{n}\sum_{i=1}^{n} w_i, \qquad
\overline{U} = \frac{1}{m}\sum_{j=1}^{m}\frac{B_j}{C_{\max}},$$

where $B_j$ is gate $j$'s total busy time and $C_{\max}$ is the makespan. Utilization is defined as zero when the makespan is zero. These metrics are calculated in `calculate_metrics`; because the denominator starts at time zero, any idle period before the first arrival is included.

In [ ]:
# These libraries support the priority queue, schedule calculations, and plots.
import heapq

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class Truck:
    """Store a truck's arrival, base service requirement, and priority."""

    def __init__(self, truck_id, arrival_time, service_time, priority, truck_type="standard"):
        if service_time < 0:
            raise ValueError("Service time must be non-negative.")

        self.id = truck_id
        self.arrival_time = arrival_time
        self.service_time = service_time
        self.priority = priority
        self.truck_type = truck_type

        # The scheduling algorithm fills these fields when it assigns the truck.
        self.assigned_gate = None
        self.start_time = None
        self.completion_time = None
        self.waiting_time = None
        self.processing_time = None

    def __lt__(self, other):
        """Order the heap by descending priority, then by arrival and ID."""
        if self.priority != other.priority:
            return self.priority > other.priority
        if self.arrival_time != other.arrival_time:
            return self.arrival_time < other.arrival_time
        return str(self.id) < str(other.id)

    def __repr__(self):
        return (
            f"Truck({self.id}, arr={self.arrival_time}, "
            f"serv={self.service_time}, pri={self.priority})"
        )


class Gate:
    """Track a gate's service-rate multiplier and scheduled workload."""

    def __init__(self, gate_id, service_rate, gate_type="standard"):
        if service_rate <= 0:
            raise ValueError("Gate service rate must be positive.")

        self.id = gate_id
        self.service_rate = service_rate
        self.gate_type = gate_type
        self.available_time = 0.0
        self.total_processed = 0
        self.total_busy_time = 0.0

    def is_available(self, current_time):
        """Return whether the gate has completed its scheduled work by this time."""
        return self.available_time <= current_time

    def assign_truck(self, truck, current_time):
        """Record a feasible service interval and update the gate's workload."""
        truck.assigned_gate = self.id
        truck.start_time = max(
            current_time,
            truck.arrival_time,
            self.available_time,
        )
        truck.processing_time = truck.service_time / self.service_rate
        truck.completion_time = truck.start_time + truck.processing_time
        truck.waiting_time = truck.start_time - truck.arrival_time

        self.available_time = truck.completion_time
        self.total_processed += 1
        self.total_busy_time += truck.processing_time

    def get_utilization(self, total_time):
        """Return the fraction of the stated time horizon spent processing trucks."""
        return self.total_busy_time / total_time if total_time > 0 else 0.0

In [ ]:
class ListSchedulingAlgorithm:
    """Schedule arrived trucks by priority and assign each to a scored gate."""

    def __init__(self, gates, priority_weight=0.6, wait_weight=0.4):
        if not gates:
            raise ValueError("At least one gate is required.")

        self.gates = gates
        self.priority_weight = priority_weight
        self.wait_weight = wait_weight

        # Arrival events and the active priority queue serve different purposes:
        # the first releases trucks over time, while the second ranks arrived trucks.
        self.arrival_trucks = []
        self.truck_queue = []
        self.processed_trucks = []
        self.current_time = 0.0

        self.makespan = 0.0
        self.avg_waiting_time = 0.0
        self.avg_gate_utilization = 0.0

    def add_truck(self, truck):
        """Register a truck for release into the priority queue at its arrival."""
        self.arrival_trucks.append(truck)

    def calculate_gate_score(self, gate, truck, current_time):
        """Calculate the heuristic score for assigning a truck to a gate."""
        priority_cost = (11 - truck.priority) * self.priority_weight
        wait_time = max(0.0, gate.available_time - current_time)
        wait_cost = wait_time * self.wait_weight
        efficiency_cost = (1.0 / gate.service_rate) * 0.1
        return priority_cost + wait_cost + efficiency_cost

    def select_best_gate(self, truck, current_time):
        """Choose the gate with the lowest score; gate order breaks exact ties."""
        return min(
            self.gates,
            key=lambda gate: self.calculate_gate_score(gate, truck, current_time),
        )

    def process_arrivals(self, arrival_list):
        """Convert input records into truck objects for the scheduling run."""
        for truck_info in arrival_list:
            truck = Truck(
                truck_id=truck_info["id"],
                arrival_time=truck_info["arrival"],
                service_time=truck_info["service"],
                priority=truck_info["priority"],
                truck_type=truck_info.get("type", "standard"),
            )
            self.add_truck(truck)

    def run_simulation(self):
        """Release arrivals over time and schedule every registered truck."""
        self.truck_queue = []
        self.processed_trucks = []
        self.current_time = 0.0

        # Reset state so a repeated run starts from the same unscheduled instance.
        for gate in self.gates:
            gate.available_time = 0.0
            gate.total_processed = 0
            gate.total_busy_time = 0.0

        for truck in self.arrival_trucks:
            truck.assigned_gate = None
            truck.start_time = None
            truck.completion_time = None
            truck.waiting_time = None
            truck.processing_time = None

        arrival_events = sorted(
            self.arrival_trucks,
            key=lambda truck: (truck.arrival_time, str(truck.id)),
        )
        next_arrival_index = 0

        while next_arrival_index < len(arrival_events) or self.truck_queue:
            while (
                next_arrival_index < len(arrival_events)
                and arrival_events[next_arrival_index].arrival_time <= self.current_time
            ):
                heapq.heappush(self.truck_queue, arrival_events[next_arrival_index])
                next_arrival_index += 1

            if self.truck_queue:
                truck = heapq.heappop(self.truck_queue)
                best_gate = self.select_best_gate(truck, self.current_time)
                best_gate.assign_truck(truck, self.current_time)
                self.processed_trucks.append(truck)
            else:
                # Advancing to the next release time avoids artificial time increments.
                self.current_time = arrival_events[next_arrival_index].arrival_time

        self.calculate_metrics()

    def calculate_metrics(self):
        """Calculate makespan, mean truck wait, and mean gate utilization."""
        if not self.processed_trucks:
            self.makespan = 0.0
            self.avg_waiting_time = 0.0
            self.avg_gate_utilization = 0.0
            return

        self.makespan = max(truck.completion_time for truck in self.processed_trucks)
        self.avg_waiting_time = sum(
            truck.waiting_time for truck in self.processed_trucks
        ) / len(self.processed_trucks)
        self.avg_gate_utilization = sum(
            gate.get_utilization(self.makespan) for gate in self.gates
        ) / len(self.gates)

    def print_schedule_results(self):
        """Print schedule-level metrics and each truck's assigned interval."""
        print("=" * 60)
        print("GATE ENTRY LIST SCHEDULING RESULTS")
        print("=" * 60)
        print(f"\nTotal Trucks Processed: {len(self.processed_trucks)}")
        print(f"Makespan: {self.makespan:.2f} minutes")
        print(f"Average Waiting Time: {self.avg_waiting_time:.2f} minutes")
        print(f"Average Gate Utilization: {self.avg_gate_utilization * 100:.1f}%")

        print("\nDetailed Gate Performance:")
        for gate in self.gates:
            utilization = gate.get_utilization(self.makespan)
            print(
                f"Gate {gate.id}: {gate.total_processed} trucks, "
                f"{utilization * 100:.1f}% utilization"
            )

        print("\nTruck Processing Details:")
        for truck in sorted(
            self.processed_trucks,
            key=lambda item: (item.start_time, str(item.id)),
        ):
            print(
                f"Truck {truck.id}: Arrival={truck.arrival_time:.1f}, "
                f"Wait={truck.waiting_time:.1f}, Start={truck.start_time:.1f}, "
                f"Complete={truck.completion_time:.1f}"
            )

    def visualize_schedule(self):
        """Plot gate assignments, utilization, waiting times, and priorities."""
        if not self.processed_trucks:
            print("No scheduled trucks are available to visualize.")
            return

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle("List Scheduling Algorithm Results", fontsize=16, fontweight="bold")

        ax1 = axes[0, 0]
        colors = plt.cm.Set3(np.linspace(0, 1, len(self.gates)))
        for gate_index, gate in enumerate(self.gates):
            gate_trucks = [
                truck
                for truck in self.processed_trucks
                if truck.assigned_gate == gate.id
            ]
            for truck in gate_trucks:
                ax1.barh(
                    gate_index,
                    truck.processing_time,
                    left=truck.start_time,
                    color=colors[gate_index],
                    alpha=0.7,
                    edgecolor="black",
                )
                ax1.text(
                    truck.start_time + truck.processing_time / 2,
                    gate_index,
                    f"T{truck.id}",
                    ha="center",
                    va="center",
                    fontweight="bold",
                )

        ax1.set_xlabel("Time (minutes)")
        ax1.set_ylabel("Gate")
        ax1.set_title("Truck Processing Schedule (Gantt Chart)", fontweight="bold")
        ax1.set_yticks(range(len(self.gates)))
        ax1.set_yticklabels([f"Gate {gate.id}" for gate in self.gates])
        ax1.grid(True, alpha=0.3)

        ax2 = axes[0, 1]
        gate_names = [f"Gate {gate.id}" for gate in self.gates]
        utilizations = [
            gate.get_utilization(self.makespan) * 100 for gate in self.gates
        ]
        bars = ax2.bar(gate_names, utilizations, color="lightgreen", alpha=0.8)
        ax2.set_title("Gate Utilization Rates", fontweight="bold")
        ax2.set_ylabel("Utilization (%)")
        ax2.grid(True, alpha=0.3)
        for bar, utilization in zip(bars, utilizations):
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1,
                f"{utilization:.1f}%",
                ha="center",
                va="bottom",
            )

        ax3 = axes[1, 0]
        ordered_trucks = sorted(
            self.processed_trucks,
            key=lambda truck: (truck.start_time, str(truck.id)),
        )
        truck_ids = [f"T{truck.id}" for truck in ordered_trucks]
        waiting_times = [truck.waiting_time for truck in ordered_trucks]
        ax3.bar(truck_ids, waiting_times, color="salmon", alpha=0.8)
        ax3.set_title("Truck Waiting Times", fontweight="bold")
        ax3.set_xlabel("Truck")
        ax3.set_ylabel("Waiting Time (minutes)")
        ax3.grid(True, alpha=0.3)
        ax3.tick_params(axis="x", rotation=45)

        ax4 = axes[1, 1]
        priorities = [truck.priority for truck in self.processed_trucks]
        waiting_times = [truck.waiting_time for truck in self.processed_trucks]
        scatter = ax4.scatter(
            priorities,
            waiting_times,
            c=waiting_times,
            cmap="viridis",
            alpha=0.7,
            s=100,
        )
        ax4.set_title("Priority vs. Waiting Time", fontweight="bold")
        ax4.set_xlabel("Priority (higher is more urgent)")
        ax4.set_ylabel("Waiting Time (minutes)")
        ax4.grid(True, alpha=0.3)
        fig.colorbar(scatter, ax=ax4, label="Waiting Time (minutes)")

        plt.tight_layout()
        plt.show()

    def compare_with_fcfs(self, arrival_list):
        """Compare the priority schedule with an arrival-order FCFS baseline."""
        fcfs_gates = [
            Gate(gate.id, gate.service_rate, gate.gate_type) for gate in self.gates
        ]
        fcfs_scheduler = ListSchedulingAlgorithm(
            fcfs_gates,
            priority_weight=0.0,
            wait_weight=1.0,
        )
        fcfs_arrivals = [
            {**truck_info, "priority": 1} for truck_info in arrival_list
        ]
        fcfs_scheduler.process_arrivals(fcfs_arrivals)
        fcfs_scheduler.run_simulation()

        print("\n" + "=" * 50)
        print("COMPARISON WITH FCFS BASELINE")
        print("=" * 50)
        print("\nPriority List Scheduling:")
        print(f"  - Makespan: {self.makespan:.2f} minutes")
        print(f"  - Average Waiting Time: {self.avg_waiting_time:.2f} minutes")
        print(f"  - Average Gate Utilization: {self.avg_gate_utilization * 100:.1f}%")

        print("\nFCFS Scheduling:")
        print(f"  - Makespan: {fcfs_scheduler.makespan:.2f} minutes")
        print(f"  - Average Waiting Time: {fcfs_scheduler.avg_waiting_time:.2f} minutes")
        print(
            f"  - Average Gate Utilization: "
            f"{fcfs_scheduler.avg_gate_utilization * 100:.1f}%"
        )

        makespan_change = None
        if fcfs_scheduler.makespan != 0:
            makespan_change = (
                (fcfs_scheduler.makespan - self.makespan)
                / fcfs_scheduler.makespan
                * 100
            )

        waiting_change = None
        if fcfs_scheduler.avg_waiting_time != 0:
            waiting_change = (
                (fcfs_scheduler.avg_waiting_time - self.avg_waiting_time)
                / fcfs_scheduler.avg_waiting_time
                * 100
            )

        print("\nPriority Schedule Improvement Relative to FCFS:")
        if makespan_change is None:
            print("  - Makespan: N/A (FCFS makespan is zero)")
        else:
            print(f"  - Makespan: {makespan_change:+.1f}%")
        if waiting_change is None:
            print("  - Waiting Time: N/A (FCFS average waiting time is zero)")
        else:
            print(f"  - Waiting Time: {waiting_change:+.1f}%")

In [ ]:
# The example uses base service times in minutes and gate-rate multipliers.
print("Concrete Example: List Scheduling with 8 Trucks and 3 Gates")
print("=" * 60)

gates = [
    Gate(gate_id=0, service_rate=1.0),
    Gate(gate_id=1, service_rate=1.2),
    Gate(gate_id=2, service_rate=0.9),
]

truck_arrivals = [
    {"id": 1, "arrival": 0.0, "service": 8.0, "priority": 10},
    {"id": 2, "arrival": 2.0, "service": 5.0, "priority": 8},
    {"id": 3, "arrival": 3.0, "service": 6.3, "priority": 6},
    {"id": 4, "arrival": 5.0, "service": 8.8, "priority": 4},
    {"id": 5, "arrival": 7.0, "service": 3.3, "priority": 9},
    {"id": 6, "arrival": 10.0, "service": 6.8, "priority": 7},
    {"id": 7, "arrival": 12.0, "service": 4.3, "priority": 8},
    {"id": 8, "arrival": 15.0, "service": 7.0, "priority": 5},
]

scheduler = ListSchedulingAlgorithm(
    gates,
    priority_weight=0.6,
    wait_weight=0.4,
)
scheduler.process_arrivals(truck_arrivals)
scheduler.run_simulation()

assert len(scheduler.processed_trucks) == len(truck_arrivals)
for truck in scheduler.processed_trucks:
    assert truck.start_time >= truck.arrival_time
    assert truck.completion_time >= truck.start_time
    assert truck.assigned_gate in [gate.id for gate in scheduler.gates]

scheduler.print_schedule_results()
scheduler.visualize_schedule()
scheduler.compare_with_fcfs(truck_arrivals)

In [ ]:
# These checks connect the reported metrics to the schedule represented by the trucks.
assert len(scheduler.processed_trucks) == len(truck_arrivals)
assert all(truck.waiting_time >= 0 for truck in scheduler.processed_trucks)

scheduled_busy_time = sum(
    truck.processing_time for truck in scheduler.processed_trucks
)
gate_busy_time = sum(gate.total_busy_time for gate in scheduler.gates)
assert np.isclose(scheduled_busy_time, gate_busy_time)

if scheduler.processed_trucks:
    assert np.isclose(
        scheduler.makespan,
        max(truck.completion_time for truck in scheduler.processed_trucks),
    )

print("Schedule checks passed: all trucks are assigned and metric totals are consistent.")

## Interpreting the results and the heuristic

The schedule table and Gantt chart show when each truck reaches its assigned gate, begins service, and completes service. A truck's waiting time is the interval between arrival and start of service. Gate utilization is each gate's scheduled busy time divided by the makespan; the average utilization is the mean across gates, not the fraction of trucks served by a particular gate.

The priority rule applies only among trucks that have arrived by the current decision time. A high-priority truck that has not arrived cannot be selected early. When priorities tie, earlier arrival is selected first, followed by the string representation of the truck ID. Gate selection then minimizes the stated score; if gate scores tie, the first gate in the supplied gate list is selected.

The FCFS comparison uses the same arrival and service data and the same gate-assignment score, but gives all trucks equal priority so that arrival order determines the queue order. The printed percentage is the relative reduction in the priority schedule's metric compared with FCFS; a negative value means the priority schedule has a larger value for that metric. A zero FCFS denominator makes a percentage undefined, so the comparison reports `N/A` in that case. Neither schedule is guaranteed to be optimal.

### Computational cost
Sorting the $n$ arrival events takes $O(n\log n)$ time, and releasing and removing trucks from the priority queue takes $O(n\log n)$ time in total. Each truck's gate selection scans all $m$ gates, taking $O(nm)$ time. The overall running time is therefore $O(n\log n + nm)$, with $O(n+m)$ space for trucks, queue entries, and gates. These bounds describe this implementation; they do not establish solution quality or an approximation ratio.

### Strengths and limitations
- The priority queue makes the urgency rule explicit, while arrival events prevent trucks from being scheduled before they arrive.
- Gate availability and service-rate multipliers are represented directly in each assignment decision.
- The heuristic produces a feasible schedule for the stated assumptions, but local score-based choices do not guarantee a globally optimal schedule.
- Results depend on the scoring rule, its weights, the tie-breaking rules, and the supplied gate order. The score components are heuristic quantities rather than a calibrated operational cost.
- The deterministic example does not represent uncertain arrivals, service-time variation, gate failures, or operational compatibility constraints.